# 3D Paper Airplane Simulator - Quickstart (Colab)

このノートブックは Google Colab 上で紙飛行機シミュレーションを手軽に試すためのものです。
環境構築なしで、以下が実行できます：

- 依存関係のインストール（CuPy自動検出、GPUなしでもCPUで動作）
- 単機シミュレーションの実行と3D軌跡の可視化
- パラメータ変更による飛行特性の比較
- 遺伝的アルゴリズムによる最適化探索のデモ

---

In [ ]:
# 1. リポジトリのクローンと依存関係インストール
import sys
import subprocess
import os

repo_url = "https://github.com/minetan1229/air_simulation.git"
repo_dir = "air_simulation"

if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url}...")
    subprocess.run(["git", "clone", "--depth", "1", repo_url], check=True)
else:
    print("Repository already exists, pulling latest...")
    subprocess.run(["git", "pull"], cwd=repo_dir, check=True)

os.chdir(repo_dir)
sys.path.insert(0, ".")

# 依存関係インストール
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)

# CuPyの自動インストール（GPUがある場合のみ）
try:
    import cupy
    print(f"CuPy already available: {cupy.__version__}")
except ImportError:
    print("CuPy not found, attempting to install for CUDA 12.x...")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "cupy-cuda12x"], check=True)
        import cupy
        print(f"CuPy installed: {cupy.__version__}")
    except Exception as e:
        print(f"CuPy install failed (will use CPU fallback): {e}")

print("Setup complete!")

In [ ]:
# 2. 基本的な単機シミュレーション実行
from physics.flight_sim import simulate_flight
from physics.aero_kernel import AircraftParams
import numpy as np

# デフォルト機体パラメータ（標準的な紙飛行機）
params = AircraftParams(
    mass=0.008,           # kg (8g)
    span=0.25,            # m (翼幅25cm)
    chord=0.05,           # m (弦長5cm)
    aspect_ratio=5.0,     # アスペクト比
    cd0=0.02,             # 零揚力抗力係数
    cl_alpha=2.0 * np.pi, # 揚力傾斜 (rad^-1)
    cl_max=1.2,           # 最大揚力係数
    alpha_stall=15.0,     # 失速迎角
    re_ref=50000.0        # 基準レイノルズ数
)

# 初期条件
initial_state = {
    "position": [0.0, 0.0, 10.0],  # x, y, z (NED座標系、zは下向き正)
    "velocity": [10.0, 0.0, 0.0],  # u, v, w (m/s)
    "quaternion": [1.0, 0.0, 0.0, 0.0],  # w, x, y, z
    "angular_velocity": [0.0, 0.0, 0.0]  # p, q, r (rad/s)
}

# シミュレーション実行 (10秒、dt=0.01s)
print("Running simulation...")
result = simulate_flight(
    params=params,
    initial_state=initial_state,
    duration=10.0,
    dt=0.01,
    wind=None  # 無風
)

print(f"Steps: {len(result['time'])}")
print(f"Final position: {result['position'][-1]}")
print(f"Final altitude: {-result['position'][-1][2]:.2f} m" )  # NED->高度
print(f"Flight time: {result['time'][-1]:.2f} s")

In [ ]:
# 3. 3D軌跡の可視化 (matplotlib)
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

pos = np.array(result['position'])  # [N, 3] in NED
time = np.array(result['time'])

fig = plt.figure(figsize=(12, 4))

# 3D軌跡
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(pos[:, 0], pos[:, 1], -pos[:, 2], 'b-', linewidth=1, alpha=0.7)
ax1.scatter([pos[0, 0]], [pos[0, 1]], [-pos[0, 2]], c='green', s=50, label='Start')
ax1.scatter([pos[-1, 0]], [pos[-1, 1]], [-pos[-1, 2]], c='red', s=50, label='End')
ax1.set_xlabel('North (m)')
ax1.set_ylabel('East (m)')
ax1.set_zlabel('Altitude (m)')
ax1.set_title('3D Flight Trajectory')
ax1.legend()
ax1.set_box_aspect([1, 1, 0.5])

# 高度-時間
ax2 = fig.add_subplot(132)
ax2.plot(time, -pos[:, 2], 'b-')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Altitude (m)')
ax2.set_title('Altitude vs Time')
ax2.grid(True, alpha=0.3)

# 速度-時間
vel = np.array(result['velocity'])  # [N, 3]
speed = np.linalg.norm(vel, axis=1)
ax3 = fig.add_subplot(133)
ax3.plot(time, speed, 'b-')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Speed (m/s)')
ax3.set_title('Speed vs Time')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 4. パラメータ変更による比較（投げ角度・初速度）
def run_simulation(v0, theta_deg, params, duration=8.0):
    """指定した初速度・投げ角度でシミュレーション"""
    theta = np.deg2rad(theta_deg)
    initial_state = {
        "position": [0.0, 0.0, 10.0],
        "velocity": [v0 * np.cos(theta), 0.0, -v0 * np.sin(theta)],
        "quaternion": [1.0, 0.0, 0.0, 0.0],
        "angular_velocity": [0.0, 0.0, 0.0]
    }
    return simulate_flight(params, initial_state, duration=duration, dt=0.01, wind=None)

# 条件を変えて実行
conditions = [
    (8.0, 5, "Low speed, shallow"),
    (10.0, 10, "Medium"),
    (12.0, 15, "High speed, steep"),
    (15.0, 0, "Fast, level"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (v0, theta, label) in enumerate(conditions):
    res = run_simulation(v0, theta, params)
    pos = np.array(res['position'])
    time = np.array(res['time'])

    ax = axes[idx]
    ax.plot(pos[:, 0], -pos[:, 2], 'b-', linewidth=1.5)
    ax.set_xlabel('Distance (m)')
    ax.set_ylabel('Altitude (m)')
    ax.set_title(f'{label}: v0={v0}m/s, θ={theta}°')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

In [ ]:
# 5. 風あり条件でのシミュレーション
from physics.wind_domain import WindField, WindConfig

# 定常風 + 乱流
wind_config = WindConfig(
    mean_speed=3.0,      # m/s
    mean_direction=45.0, # 度 (北から時計回り)
    turbulence_intensity=0.15,
    reference_height=10.0
)
wind_field = WindField(wind_config)

initial_state = {
    "position": [0.0, 0.0, 10.0],
    "velocity": [10.0, 0.0, 0.0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "angular_velocity": [0.0, 0.0, 0.0]
}

print("Running with wind...")
result_wind = simulate_flight(
    params=params,
    initial_state=initial_state,
    duration=10.0,
    dt=0.01,
    wind=wind_field
)

pos_wind = np.array(result_wind['position'])
pos_nowind = np.array(result['position'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(pos_nowind[:, 0], pos_nowind[:, 1], 'b-', label='No wind', alpha=0.7)
ax1.plot(pos_wind[:, 0], pos_wind[:, 1], 'r-', label='With wind (3m/s @ 45°)', alpha=0.7)
ax1.set_xlabel('North (m)')
ax1.set_ylabel('East (m)')
ax1.set_title('Ground Track Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal', adjustable='box')

ax2.plot(-pos_nowind[:, 2], 'b-', label='No wind')
ax2.plot(-pos_wind[:, 2], 'r-', label='With wind')
ax2.set_xlabel('Step')
ax2.set_ylabel('Altitude (m)')
ax2.set_title('Altitude Profile')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. 遺伝的アルゴリズムによる最適化デモ（軽量版）
# 注: 完全版はローカル環境推奨。ここでは概念実証のみ。

try:
    from physics.ga2.population import Population
    from physics.ga2.objectives import distance_objective, time_objective
    from physics.ga2.config import GAConfig

    # 軽量設定（個体数少・世代数少）
    config = GAConfig(
        population_size=20,
        max_generations=10,
        mutation_rate=0.1,
        crossover_rate=0.8,
        elite_size=2
    )

    # パラメータ境界
    bounds = {
        "mass": (0.005, 0.020),
        "span": (0.15, 0.35),
        "chord": (0.03, 0.08),
        "cd0": (0.01, 0.05),
        "cl_alpha": (4.0, 8.0),
    }

    print("Starting GA optimization (demo)...")
    pop = Population(config, bounds)

    # 簡易評価関数（距離最大化）
    def evaluate(individual):
        p = AircraftParams(
            mass=individual["mass"],
            span=individual["span"],
            chord=individual["chord"],
            aspect_ratio=individual["span"] / individual["chord"],
            cd0=individual["cd0"],
            cl_alpha=individual["cl_alpha"],
            cl_max=1.2,
            alpha_stall=15.0,
            re_ref=50000.0
        )
        init = {
            "position": [0.0, 0.0, 10.0],
            "velocity": [10.0, 0.0, 0.0],
            "quaternion": [1.0, 0.0, 0.0, 0.0],
            "angular_velocity": [0.0, 0.0, 0.0]
        }
        res = simulate_flight(p, init, duration=8.0, dt=0.02, wind=None)
        final_pos = np.array(res['position'][-1])
        distance = np.sqrt(final_pos[0]**2 + final_pos[1]**2)
        return distance

    # 数世代だけ実行
    for gen in range(5):
        pop.evaluate_all(evaluate)
        best = pop.get_best()
        print(f"Gen {gen+1}: Best distance = {best.fitness:.2f} m")
        pop.evolve()

    print("\nOptimization demo complete.")
    print(f"Best individual: {best.genes}")
    print(f"Best fitness: {best.fitness:.2f} m")

except Exception as e:
    print(f"GA demo skipped (requires full physics.ga2 module): {e}")
    print("For full GA optimization, run locally with: python -m physics.ga2.cli")

---

## 次のステップ

- **ローカルで本格運用**: `git clone` して `python app.py` でWeb UI付きフル機能版を起動
- **パラメータ探索**: `examples/quick_simulation.py` をベースに独自の最適化スクリプトを作成
- **物理モデルの理解**: `docs/PHYSICS_MODELS.md` で数式詳細を確認
- **API活用**: `http://localhost:5000/docs` (Swagger UI) でREST APIを試す

## トラブルシューティング

- **CuPyが入らない**: `pip install cupy-cuda11x` または `cupy-cuda12x` をCUDAバージョンに合わせて指定
- **メモリ不足**: Colab無料枠はGPUメモリ約12GB。個体数・世代数を減らすかCPUモードで実行
- **importエラー**: `sys.path.insert(0, ".")` でカレントディレクトリをパスに追加済み

---

*このノートブックは [air_simulation](https://github.com/minetan1229/air_simulation) リポジトリの一部です。*